# Section 1

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Imports completed successfully.")

Imports completed successfully.


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

Device: cuda
GPU: Tesla T4


In [4]:
import os

BASE_DIR = "/content/drive/MyDrive/Major_Project"

ENGINEERED_DATA_PATH = os.path.join(
    BASE_DIR,
    "dataset/engineered/engineered_dataset.csv"
)

ENGINEERED_PKL_PATH = os.path.join(
    BASE_DIR,
    "dataset/engineered/engineered_dataset.pkl"
)

LABEL_ENCODER_PATH = os.path.join(
    BASE_DIR,
    "models/label_encoder.pkl"
)

SCALER_PATH = os.path.join(
    BASE_DIR,
    "models/scaler.pkl"
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "models/hybrid_model.pth"
)

TRAINING_HISTORY_PATH = os.path.join(
    BASE_DIR,
    "models/training_history.json"
)

CHECKPOINT_PATH = os.path.join(
    BASE_DIR,
    "models/training_checkpoint.pth"
)

EVALUATION_DIR = os.path.join(
    BASE_DIR,
    "models/evaluation"
)

print("Project paths configured.")

Project paths configured.


In [5]:
required_files = {
    "Engineered CSV": ENGINEERED_DATA_PATH,
    "Engineered PKL": ENGINEERED_PKL_PATH,
    "Label Encoder": LABEL_ENCODER_PATH,
    "Scaler": SCALER_PATH,
    "Best Model": MODEL_PATH,
    "Training History": TRAINING_HISTORY_PATH,
    "Training Checkpoint": CHECKPOINT_PATH
}

for name, path in required_files.items():
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"{name}: {status}")

Engineered CSV: FOUND
Engineered PKL: FOUND
Label Encoder: FOUND
Scaler: FOUND
Best Model: FOUND
Training History: FOUND
Training Checkpoint: MISSING


# Section 2

In [6]:
import joblib

label_encoder = joblib.load(LABEL_ENCODER_PATH)

print("Label encoder loaded successfully.")
print("Number of classes:", len(label_encoder.classes_))
print("Classes:")
print(label_encoder.classes_)

Label encoder loaded successfully.
Number of classes: 17
Classes:
['BENIGN' 'DrDoS_DNS' 'DrDoS_LDAP' 'DrDoS_MSSQL' 'DrDoS_NTP'
 'DrDoS_NetBIOS' 'DrDoS_SNMP' 'DrDoS_SSDP' 'DrDoS_UDP' 'LDAP' 'MSSQL'
 'NetBIOS' 'Portmap' 'Syn' 'TFTP' 'UDP' 'UDP-lag']


In [7]:
engineered_df = pd.read_pickle(ENGINEERED_PKL_PATH)

print("Engineered dataset loaded successfully.")
print("Shape:", engineered_df.shape)
print("\nColumns:")
print(engineered_df.columns.tolist())

Engineered dataset loaded successfully.
Shape: (3384445, 49)

Columns:
['Flow Duration_Mean', 'Flow Duration_Std', 'Flow Duration_Variance', 'Flow Duration_Min', 'Flow Duration_Max', 'Flow Duration_Renyi', 'Flow Bytes/s_Mean', 'Flow Bytes/s_Std', 'Flow Bytes/s_Variance', 'Flow Bytes/s_Min', 'Flow Bytes/s_Max', 'Flow Bytes/s_Renyi', 'Flow Packets/s_Mean', 'Flow Packets/s_Std', 'Flow Packets/s_Variance', 'Flow Packets/s_Min', 'Flow Packets/s_Max', 'Flow Packets/s_Renyi', 'Packet Length Mean_Mean', 'Packet Length Mean_Std', 'Packet Length Mean_Variance', 'Packet Length Mean_Min', 'Packet Length Mean_Max', 'Packet Length Mean_Renyi', 'Packet Length Std_Mean', 'Packet Length Std_Std', 'Packet Length Std_Variance', 'Packet Length Std_Min', 'Packet Length Std_Max', 'Packet Length Std_Renyi', 'Total Fwd Packets_Mean', 'Total Fwd Packets_Std', 'Total Fwd Packets_Variance', 'Total Fwd Packets_Min', 'Total Fwd Packets_Max', 'Total Fwd Packets_Renyi', 'Total Backward Packets_Mean', 'Total Backward

In [8]:
print("Missing values:", engineered_df.isnull().sum().sum())
print("Duplicate rows:", engineered_df.duplicated().sum())

print("\nData types:")
print(engineered_df.dtypes)

print("\nFirst 5 rows:")
display(engineered_df.head())

Missing values: 0
Duplicate rows: 0

Data types:
Flow Duration_Mean                 float64
Flow Duration_Std                  float64
Flow Duration_Variance             float64
Flow Duration_Min                    int64
Flow Duration_Max                    int64
Flow Duration_Renyi                float64
Flow Bytes/s_Mean                  float64
Flow Bytes/s_Std                   float64
Flow Bytes/s_Variance              float64
Flow Bytes/s_Min                   float64
Flow Bytes/s_Max                   float64
Flow Bytes/s_Renyi                 float64
Flow Packets/s_Mean                float64
Flow Packets/s_Std                 float64
Flow Packets/s_Variance            float64
Flow Packets/s_Min                 float64
Flow Packets/s_Max                 float64
Flow Packets/s_Renyi               float64
Packet Length Mean_Mean            float64
Packet Length Mean_Std             float64
Packet Length Mean_Variance        float64
Packet Length Mean_Min             float64
Packe

,Flow Duration_Mean,Flow Duration_Std,Flow Duration_Variance,Flow Duration_Min,Flow Duration_Max,Flow Duration_Renyi,Flow Bytes/s_Mean,Flow Bytes/s_Std,Flow Bytes/s_Variance,Flow Bytes/s_Min,...,Total Backward Packets_Min,Total Backward Packets_Max,Total Backward Packets_Renyi,Average Packet Size_Mean,Average Packet Size_Std,Average Packet Size_Variance,Average Packet Size_Min,Average Packet Size_Max,Average Packet Size_Renyi,Label
0,10839530.69,2.843003e+07,8.082669e+14,1,119842965,0.595533,333457.721947,3.083312e+06,9.506815e+12,0.0,...,0,96,0.787368,119.622965,223.877889,50121.309327,0.0,1533.777778,1.270904,BENIGN
1,10505891.36,2.883378e+07,8.313870e+14,96,118514337,0.525330,42053.922346,2.446347e+05,5.984614e+10,0.0,...,0,201,0.696190,184.627261,317.521456,100819.875155,0.0,1784.444444,1.225666,BENIGN
2,12329517.29,3.187952e+07,1.016304e+15,97,118759341,0.586406,65741.641517,2.704786e+05,7.315865e+10,0.0,...,0,260,0.903468,252.496307,416.828398,173745.913406,0.0,1784.444444,1.502260,BENIGN
3,15855919.38,3.638650e+07,1.323977e+15,1,118759341,0.681770,376029.512487,3.089682e+06,9.546132e+12,0.0,...,0,260,1.081308,251.265778,418.410308,175067.185685,0.0,1784.444444,1.682233,BENIGN
4,15815469.62,3.650741e+07,1.332791e+15,1,118759341,0.647016,376991.772878,3.089569e+06,9.545437e+12,0.0,...,0,260,1.043943,254.374859,416.931181,173831.609710,0.0,1784.444444,1.531156,BENIGN


# Section 3


In [9]:
# ==========================================================
# Section 3: Frozen Baseline Model
# Cell 1: Attention Module
# ==========================================================

class Attention(nn.Module):
    """
    Attention Layer for Bi-LSTM Outputs
    """

    def __init__(self, hidden_size):
        super(Attention, self).__init__()

        self.attention = nn.Linear(hidden_size * 2, 1)

    def forward(self, lstm_output):
        """
        lstm_output shape:
        (batch_size, sequence_length, hidden_size * 2)
        """

        # Compute attention scores
        scores = self.attention(lstm_output)

        # Normalize scores
        weights = torch.softmax(scores, dim=1)

        # Context Vector
        context = torch.sum(weights * lstm_output, dim=1)

        return context, weights

In [10]:
# ==========================================================
# Cell 2: Exact Hybrid CNN-BiLSTM-Attention Model
# ==========================================================

class HybridDDoSModel(nn.Module):
    """
    Hybrid CNN + BiLSTM + Attention Network
    """

    def __init__(self,
                 input_channels=1,
                 num_classes=17):

        super(HybridDDoSModel, self).__init__()

        # -----------------------------
        # CNN Feature Extractor
        # -----------------------------
        self.cnn = nn.Sequential(

            nn.Conv1d(
                in_channels=input_channels,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        # -----------------------------
        # Bi-LSTM
        # -----------------------------
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # -----------------------------
        # Attention
        # -----------------------------
        self.attention = Attention(hidden_size=128)

        # -----------------------------
        # Classifier
        # -----------------------------
        self.classifier = nn.Sequential(

            nn.Dropout(0.3),

            nn.Linear(256, 64),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        # CNN
        x = self.cnn(x)

        # Convert to
        # (batch, sequence, channels)
        x = x.permute(0, 2, 1)

        # BiLSTM
        lstm_output, _ = self.lstm(x)

        # Attention
        context, attention_weights = self.attention(lstm_output)

        # Classification
        output = self.classifier(context)

        return output

In [11]:
# ==========================================================
# Cell 3: Load Frozen Baseline Model
# ==========================================================

model = HybridDDoSModel(
    input_channels=1,
    num_classes=17
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print("Frozen baseline model loaded successfully.")

Frozen baseline model loaded successfully.


In [12]:
# ==========================================================
# Cell 4: Model Verification
# ==========================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Total parameters:", total_params)

# Use actual 48 feature columns
feature_columns = [
    col for col in engineered_df.columns
    if col != "Label"
]

X_sample = engineered_df[
    feature_columns
].iloc[:4].values

X_sample_tensor = torch.tensor(
    X_sample,
    dtype=torch.float32
).unsqueeze(1).to(device)

print("Input shape:", X_sample_tensor.shape)

with torch.no_grad():
    output = model(X_sample_tensor)

print("Output shape:", output.shape)

Total parameters: 307346
Input shape: torch.Size([4, 1, 48])
Output shape: torch.Size([4, 17])


# Section 4


In [13]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 1: Normalize Dataset Labels
# ==========================================================

# The saved label encoder uses "UDP-lag".
# The engineered dataset contains one additional spelling: "UDPLag".
# Normalize it to the encoder's existing class name.

engineered_df["Label"] = engineered_df["Label"].replace(
    {"UDPLag": "UDP-lag"}
)

print("Unique labels after normalization:")
print(sorted(engineered_df["Label"].unique()))

print("\nNumber of classes:",
      engineered_df["Label"].nunique())

Unique labels after normalization:
['BENIGN', 'DrDoS_DNS', 'DrDoS_LDAP', 'DrDoS_MSSQL', 'DrDoS_NTP', 'DrDoS_NetBIOS', 'DrDoS_SNMP', 'DrDoS_SSDP', 'DrDoS_UDP', 'LDAP', 'MSSQL', 'NetBIOS', 'Portmap', 'Syn', 'TFTP', 'UDP', 'UDP-lag']

Number of classes: 17


In [14]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 2: Encode Labels
# ==========================================================

X = engineered_df[feature_columns].values

y = label_encoder.transform(
    engineered_df["Label"]
)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

print("\nEncoded label range:")
print("Minimum:", y.min())
print("Maximum:", y.max())

print("\nUnique encoded classes:")
print(np.unique(y))

Feature shape: (3384445, 48)
Label shape: (3384445,)

Encoded label range:
Minimum: 0
Maximum: 16

Unique encoded classes:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]


In [15]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 3: Reproduce Train/Validation/Test Split
# ==========================================================

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (2707556, 48)
Validation: (338444, 48)
Test: (338445, 48)


In [16]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 4: Create Test DataLoader
# ==========================================================

from torch.utils.data import TensorDataset, DataLoader

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
).unsqueeze(1)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False
)

print("Test DataLoader created.")
print("Test batches:", len(test_loader))

Test DataLoader created.
Test batches: 662


In [17]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 6A: Load Notebook 05 Scaler
# ==========================================================

import joblib

scaler = joblib.load(SCALER_PATH)

print("Scaler loaded successfully.")
print("Number of features:", len(scaler.mean_))

Scaler loaded successfully.
Number of features: 48


In [18]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 6B: Apply Notebook 05 Scaling
# ==========================================================

# Apply the saved StandardScaler to the test features.
# This reproduces the preprocessing used during Notebook 05 training.

X_test_scaled = scaler.transform(X_test)

print("Scaled test shape:", X_test_scaled.shape)

print("\nScaled feature statistics:")
print(
    f"Mean: {X_test_scaled.mean():.6f}"
)

print(
    f"Std : {X_test_scaled.std():.6f}"
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Scaled test shape: (338445, 48)

Scaled feature statistics:
Mean: -0.000131
Std : 0.961269


In [19]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 6C: Recreate Scaled Test DataLoader
# ==========================================================

X_test_scaled_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
).unsqueeze(1)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

test_dataset = TensorDataset(
    X_test_scaled_tensor,
    y_test_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False
)

print("Scaled test DataLoader recreated.")
print("Test batches:", len(test_loader))

Scaled test DataLoader recreated.
Test batches: 662


In [20]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 5: Generate Predictions & Confidence
# ==========================================================

all_predictions = []
all_probabilities = []
all_true_labels = []

model.eval()

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        batch_X = batch_X.to(device)

        logits = model(batch_X)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        confidence, predictions = torch.max(
            probabilities,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            confidence.cpu().numpy()
        )

        all_true_labels.extend(
            batch_y.numpy()
        )

all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)
all_true_labels = np.array(all_true_labels)

print("Predictions:", len(all_predictions))
print("Confidence values:", len(all_probabilities))
print("True labels:", len(all_true_labels))

Predictions: 338445
Confidence values: 338445
True labels: 338445


In [21]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 6: Verify Baseline Accuracy
# ==========================================================

baseline_accuracy = accuracy_score(
    all_true_labels,
    all_predictions
)

print(
    f"Baseline Test Accuracy: "
    f"{baseline_accuracy * 100:.2f}%"
)

Baseline Test Accuracy: 89.43%


In [22]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 7: Analyze Prediction Confidence
# ==========================================================

print("Prediction Confidence Statistics")
print("=" * 60)

print(
    f"Mean Confidence   : "
    f"{np.mean(all_probabilities):.4f}"
)

print(
    f"Median Confidence : "
    f"{np.median(all_probabilities):.4f}"
)

print(
    f"Minimum Confidence: "
    f"{np.min(all_probabilities):.4f}"
)

print(
    f"Maximum Confidence: "
    f"{np.max(all_probabilities):.4f}"
)

print(
    f"25th Percentile   : "
    f"{np.percentile(all_probabilities, 25):.4f}"
)

print(
    f"75th Percentile   : "
    f"{np.percentile(all_probabilities, 75):.4f}"
)

Prediction Confidence Statistics
Mean Confidence   : 0.8926
Median Confidence : 0.9997
Minimum Confidence: 0.2372
Maximum Confidence: 1.0000
25th Percentile   : 0.8335
75th Percentile   : 1.0000


In [23]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 8: Compare Confidence of Correct and Incorrect Predictions
# ==========================================================

correct_mask = (
    all_predictions == all_true_labels
)

incorrect_mask = (
    all_predictions != all_true_labels
)

correct_confidence = all_probabilities[
    correct_mask
]

incorrect_confidence = all_probabilities[
    incorrect_mask
]

print("Correct Predictions")
print("=" * 60)

print(
    f"Samples          : {len(correct_confidence):,}"
)

print(
    f"Mean Confidence  : "
    f"{np.mean(correct_confidence):.4f}"
)

print(
    f"Median Confidence: "
    f"{np.median(correct_confidence):.4f}"
)

print("\nIncorrect Predictions")
print("=" * 60)

print(
    f"Samples          : {len(incorrect_confidence):,}"
)

print(
    f"Mean Confidence  : "
    f"{np.mean(incorrect_confidence):.4f}"
)

print(
    f"Median Confidence: "
    f"{np.median(incorrect_confidence):.4f}"
)

Correct Predictions
Samples          : 302,678
Mean Confidence  : 0.9243
Median Confidence: 1.0000

Incorrect Predictions
Samples          : 35,767
Mean Confidence  : 0.6241
Median Confidence: 0.6003


In [24]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 9: Confidence-Based Error Analysis
# ==========================================================

confidence_thresholds = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

print("Incorrect Prediction Coverage by Confidence Threshold")
print("=" * 70)

for threshold in confidence_thresholds:

    low_confidence_mask = (
        all_probabilities < threshold
    )

    incorrect_low_confidence = (
        low_confidence_mask & incorrect_mask
    )

    total_low_confidence = (
        low_confidence_mask.sum()
    )

    captured_errors = (
        incorrect_low_confidence.sum()
    )

    error_capture_rate = (
        captured_errors / incorrect_mask.sum()
        if incorrect_mask.sum() > 0
        else 0
    )

    print(
        f"Threshold < {threshold:.2f} | "
        f"Low-confidence samples: {total_low_confidence:6,} | "
        f"Incorrect captured: {captured_errors:6,} | "
        f"Error capture: {error_capture_rate * 100:6.2f}%"
    )

Incorrect Prediction Coverage by Confidence Threshold
Threshold < 0.50 | Low-confidence samples: 15,537 | Incorrect captured:  8,665 | Error capture:  24.23%
Threshold < 0.60 | Low-confidence samples: 36,201 | Incorrect captured: 17,855 | Error capture:  49.92%
Threshold < 0.70 | Low-confidence samples: 56,215 | Incorrect captured: 24,631 | Error capture:  68.87%
Threshold < 0.80 | Low-confidence samples: 76,591 | Incorrect captured: 29,629 | Error capture:  82.84%
Threshold < 0.90 | Low-confidence samples: 104,397 | Incorrect captured: 33,535 | Error capture:  93.76%


In [25]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 10: Extract Rényi Entropy Features
# ==========================================================

# Identify all Rényi entropy features
renyi_columns = [
    col for col in feature_columns
    if "_Renyi" in col
]

print("Rényi Entropy Features")
print("=" * 60)

for column in renyi_columns:
    print(column)

print("\nNumber of Rényi features:", len(renyi_columns))

Rényi Entropy Features
Flow Duration_Renyi
Flow Bytes/s_Renyi
Flow Packets/s_Renyi
Packet Length Mean_Renyi
Packet Length Std_Renyi
Total Fwd Packets_Renyi
Total Backward Packets_Renyi
Average Packet Size_Renyi

Number of Rényi features: 8


In [26]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 11: Rényi Statistics by Prediction Correctness
# ==========================================================

# Extract Rényi features from the test set.
# X_test contains the same feature ordering as feature_columns.

renyi_indices = [
    feature_columns.index(col)
    for col in renyi_columns
]

X_test_renyi = X_test[:, renyi_indices]

correct_test_renyi = X_test_renyi[
    correct_mask
]

incorrect_test_renyi = X_test_renyi[
    incorrect_mask
]

print("Rényi Statistics")
print("=" * 80)

for i, column in enumerate(renyi_columns):

    correct_mean = np.mean(
        correct_test_renyi[:, i]
    )

    incorrect_mean = np.mean(
        incorrect_test_renyi[:, i]
    )

    correct_median = np.median(
        correct_test_renyi[:, i]
    )

    incorrect_median = np.median(
        incorrect_test_renyi[:, i]
    )

    print(f"\n{column}")
    print(
        f"  Correct   → "
        f"Mean: {correct_mean:.4f}, "
        f"Median: {correct_median:.4f}"
    )
    print(
        f"  Incorrect → "
        f"Mean: {incorrect_mean:.4f}, "
        f"Median: {incorrect_median:.4f}"
    )

Rényi Statistics

Flow Duration_Renyi
  Correct   → Mean: 0.5342, Median: 0.4139
  Incorrect → Mean: 0.8390, Median: 0.6597

Flow Bytes/s_Renyi
  Correct   → Mean: 1.1168, Median: 1.0297
  Incorrect → Mean: 1.5608, Median: 1.3255

Flow Packets/s_Renyi
  Correct   → Mean: 0.7442, Median: 0.7486
  Incorrect → Mean: 0.8788, Median: 0.9280

Packet Length Mean_Renyi
  Correct   → Mean: 0.7181, Median: 0.2983
  Incorrect → Mean: 1.7903, Median: 2.1884

Packet Length Std_Renyi
  Correct   → Mean: 0.1632, Median: 0.0000
  Incorrect → Mean: 0.6808, Median: 0.0000

Total Fwd Packets_Renyi
  Correct   → Mean: 0.3500, Median: 0.0289
  Incorrect → Mean: 0.6750, Median: 0.0580

Total Backward Packets_Renyi
  Correct   → Mean: 0.0901, Median: 0.0000
  Incorrect → Mean: 0.0088, Median: 0.0000

Average Packet Size_Renyi
  Correct   → Mean: 0.9516, Median: 0.5726
  Incorrect → Mean: 1.8771, Median: 2.3248


In [27]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 12: Measure Rényi Separation
# ==========================================================

print("Rényi Separation Between Correct and Incorrect Predictions")
print("=" * 80)

renyi_separation = {}

for i, column in enumerate(renyi_columns):

    correct_mean = np.mean(
        correct_test_renyi[:, i]
    )

    incorrect_mean = np.mean(
        incorrect_test_renyi[:, i]
    )

    # Absolute difference between the two groups
    separation = abs(
        incorrect_mean - correct_mean
    )

    renyi_separation[column] = separation

    print(
        f"{column:<35} "
        f"Separation: {separation:.4f}"
    )

print("\nRanked Rényi Features")
print("=" * 80)

ranked_features = sorted(
    renyi_separation.items(),
    key=lambda x: x[1],
    reverse=True
)

for rank, (feature, value) in enumerate(
    ranked_features,
    start=1
):
    print(
        f"{rank}. {feature:<35} "
        f"{value:.4f}"
    )

Rényi Separation Between Correct and Incorrect Predictions
Flow Duration_Renyi                 Separation: 0.3048
Flow Bytes/s_Renyi                  Separation: 0.4441
Flow Packets/s_Renyi                Separation: 0.1347
Packet Length Mean_Renyi            Separation: 1.0722
Packet Length Std_Renyi             Separation: 0.5176
Total Fwd Packets_Renyi             Separation: 0.3250
Total Backward Packets_Renyi        Separation: 0.0813
Average Packet Size_Renyi           Separation: 0.9255

Ranked Rényi Features
1. Packet Length Mean_Renyi            1.0722
2. Average Packet Size_Renyi           0.9255
3. Packet Length Std_Renyi             0.5176
4. Flow Bytes/s_Renyi                  0.4441
5. Total Fwd Packets_Renyi             0.3250
6. Flow Duration_Renyi                 0.3048
7. Flow Packets/s_Renyi                0.1347
8. Total Backward Packets_Renyi        0.0813


In [28]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 13: Build Normalized Rényi Evidence Score
# ==========================================================

from sklearn.preprocessing import StandardScaler

# Convert the 8 Rényi features into a standardized representation.
# This prevents features with larger numerical ranges from
# dominating the combined evidence score.

renyi_scaler = StandardScaler()

renyi_standardized = renyi_scaler.fit_transform(
    X_test_renyi
)

# Use the magnitude of the standardized values as an
# initial evidence measure.
renyi_evidence_score = np.mean(
    np.abs(renyi_standardized),
    axis=1
)

print("Rényi evidence score generated.")

print("\nStatistics")
print("=" * 60)

print(
    f"Mean   : "
    f"{renyi_evidence_score.mean():.4f}"
)

print(
    f"Median : "
    f"{np.median(renyi_evidence_score):.4f}"
)

print(
    f"Minimum: "
    f"{renyi_evidence_score.min():.4f}"
)

print(
    f"Maximum: "
    f"{renyi_evidence_score.max():.4f}"
)

Rényi evidence score generated.

Statistics
Mean   : 0.7665
Median : 0.6925
Minimum: 0.1968
Maximum: 3.8232


In [29]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 14: Compare Rényi Evidence by Prediction Correctness
# ==========================================================

correct_evidence = renyi_evidence_score[
    correct_mask
]

incorrect_evidence = renyi_evidence_score[
    incorrect_mask
]

print("Rényi Evidence by Prediction Correctness")
print("=" * 70)

print("\nCorrect Predictions")
print("-" * 70)

print(
    f"Samples           : "
    f"{len(correct_evidence):,}"
)

print(
    f"Mean Evidence     : "
    f"{np.mean(correct_evidence):.4f}"
)

print(
    f"Median Evidence   : "
    f"{np.median(correct_evidence):.4f}"
)

print(
    f"75th Percentile   : "
    f"{np.percentile(correct_evidence, 75):.4f}"
)

print("\nIncorrect Predictions")
print("-" * 70)

print(
    f"Samples           : "
    f"{len(incorrect_evidence):,}"
)

print(
    f"Mean Evidence     : "
    f"{np.mean(incorrect_evidence):.4f}"
)

print(
    f"Median Evidence   : "
    f"{np.median(incorrect_evidence):.4f}"
)

print(
    f"75th Percentile   : "
    f"{np.percentile(incorrect_evidence, 75):.4f}"
)

Rényi Evidence by Prediction Correctness

Correct Predictions
----------------------------------------------------------------------
Samples           : 302,678
Mean Evidence     : 0.7430
Median Evidence   : 0.6680
75th Percentile   : 0.9109

Incorrect Predictions
----------------------------------------------------------------------
Samples           : 35,767
Mean Evidence     : 0.9655
Median Evidence   : 1.0393
75th Percentile   : 1.2486


In [30]:
# ==========================================================
# Section 4: Baseline Prediction & Confidence Analysis
# Cell 15: Analyze Combined Confidence-Rényi Relationship
# ==========================================================

# Lower confidence indicates less reliable model prediction.
# Higher Rényi evidence indicates stronger statistical deviation
# in the current exploratory analysis.

# Normalize confidence so that higher values indicate
# greater uncertainty.
confidence_uncertainty = 1.0 - all_probabilities

print("Combined Evidence Analysis")
print("=" * 70)

print("\nCorrect Predictions")
print("-" * 70)

print(
    f"Mean Confidence Uncertainty : "
    f"{confidence_uncertainty[correct_mask].mean():.4f}"
)

print(
    f"Mean Rényi Evidence         : "
    f"{renyi_evidence_score[correct_mask].mean():.4f}"
)

print("\nIncorrect Predictions")
print("-" * 70)

print(
    f"Mean Confidence Uncertainty : "
    f"{confidence_uncertainty[incorrect_mask].mean():.4f}"
)

print(
    f"Mean Rényi Evidence         : "
    f"{renyi_evidence_score[incorrect_mask].mean():.4f}"
)

Combined Evidence Analysis

Correct Predictions
----------------------------------------------------------------------
Mean Confidence Uncertainty : 0.0757
Mean Rényi Evidence         : 0.7430

Incorrect Predictions
----------------------------------------------------------------------
Mean Confidence Uncertainty : 0.3759
Mean Rényi Evidence         : 0.9655


# Section 5

In [31]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 1: Define Evidence Components
# ==========================================================

# Confidence uncertainty:
# Lower model confidence = higher uncertainty.
confidence_uncertainty = 1.0 - all_probabilities

# Rényi evidence:
# Higher standardized Rényi deviation = stronger
# statistical evidence in the current traffic window.
#
# The exploratory Rényi score was already calculated
# from the eight engineered Rényi features.
statistical_evidence = renyi_evidence_score

print("AnomalyFlow evidence components prepared.")
print("=" * 70)

print(
    f"Confidence Uncertainty → "
    f"Mean: {confidence_uncertainty.mean():.4f}"
)

print(
    f"Rényi Evidence         → "
    f"Mean: {statistical_evidence.mean():.4f}"
)

AnomalyFlow evidence components prepared.
Confidence Uncertainty → Mean: 0.1074
Rényi Evidence         → Mean: 0.7665


In [32]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 2: Normalize Evidence Signals
# ==========================================================

from sklearn.preprocessing import MinMaxScaler

# Reshape signals for sklearn
confidence_values = confidence_uncertainty.reshape(-1, 1)
renyi_values = statistical_evidence.reshape(-1, 1)

# Normalize both signals to [0, 1]
confidence_scaler = MinMaxScaler()
renyi_evidence_scaler = MinMaxScaler()

confidence_normalized = (
    confidence_scaler.fit_transform(
        confidence_values
    ).flatten()
)

renyi_normalized = (
    renyi_evidence_scaler.fit_transform(
        renyi_values
    ).flatten()
)

print("Evidence normalization completed.")
print("=" * 70)

print(
    f"Confidence Uncertainty → "
    f"Min: {confidence_normalized.min():.4f}, "
    f"Max: {confidence_normalized.max():.4f}"
)

print(
    f"Rényi Evidence         → "
    f"Min: {renyi_normalized.min():.4f}, "
    f"Max: {renyi_normalized.max():.4f}"
)

Evidence normalization completed.
Confidence Uncertainty → Min: 0.0000, Max: 1.0000
Rényi Evidence         → Min: 0.0000, Max: 1.0000


In [33]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 3: Combine Evidence Signals
# ==========================================================

# Combine model uncertainty and statistical evidence.
#
# Equal weighting is used initially so that neither signal
# is given an arbitrary advantage before evaluation.

CONFIDENCE_WEIGHT = 0.5
RENYI_WEIGHT = 0.5

anomalyflow_score = (
    CONFIDENCE_WEIGHT * confidence_normalized
    +
    RENYI_WEIGHT * renyi_normalized
)

print("AnomalyFlow score generated.")
print("=" * 70)

print(
    f"Mean   : {anomalyflow_score.mean():.4f}"
)

print(
    f"Median : {np.median(anomalyflow_score):.4f}"
)

print(
    f"Minimum: {anomalyflow_score.min():.4f}"
)

print(
    f"Maximum: {anomalyflow_score.max():.4f}"
)

AnomalyFlow score generated.
Mean   : 0.1490
Median : 0.0886
Minimum: 0.0068
Maximum: 0.6313


In [34]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 4: Evaluate Combined Score Separation
# ==========================================================

correct_scores = anomalyflow_score[
    correct_mask
]

incorrect_scores = anomalyflow_score[
    incorrect_mask
]

print("AnomalyFlow Score by Prediction Correctness")
print("=" * 70)

print("\nCorrect Predictions")
print("-" * 70)

print(
    f"Mean Score   : "
    f"{correct_scores.mean():.4f}"
)

print(
    f"Median Score : "
    f"{np.median(correct_scores):.4f}"
)

print(
    f"75th Percentile: "
    f"{np.percentile(correct_scores, 75):.4f}"
)

print("\nIncorrect Predictions")
print("-" * 70)

print(
    f"Mean Score   : "
    f"{incorrect_scores.mean():.4f}"
)

print(
    f"Median Score : "
    f"{np.median(incorrect_scores):.4f}"
)

print(
    f"75th Percentile: "
    f"{np.percentile(incorrect_scores, 75):.4f}"
)

AnomalyFlow Score by Prediction Correctness

Correct Predictions
----------------------------------------------------------------------
Mean Score   : 0.1249
Median Score : 0.0784
75th Percentile: 0.1510

Incorrect Predictions
----------------------------------------------------------------------
Mean Score   : 0.3524
Median Score : 0.3641
75th Percentile: 0.4593


In [35]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 5: Analyze Candidate Score Thresholds
# ==========================================================

score_thresholds = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

total_incorrect = incorrect_mask.sum()
total_correct = correct_mask.sum()

print("AnomalyFlow Score Threshold Analysis")
print("=" * 100)

print(
    f"{'Threshold':<12}"
    f"{'Flagged':<12}"
    f"{'Errors Caught':<16}"
    f"{'Error Recall':<15}"
    f"{'Correct Flagged':<18}"
    f"{'Correct Flag Rate':<18}"
)

print("-" * 100)

for threshold in score_thresholds:

    flagged = anomalyflow_score >= threshold

    errors_caught = (
        flagged & incorrect_mask
    ).sum()

    correct_flagged = (
        flagged & correct_mask
    ).sum()

    error_recall = (
        errors_caught / total_incorrect
    )

    correct_flag_rate = (
        correct_flagged / total_correct
    )

    print(
        f"{threshold:<12.2f}"
        f"{flagged.sum():<12,}"
        f"{errors_caught:<16,}"
        f"{error_recall * 100:<15.2f}%"
        f"{correct_flagged:<18,}"
        f"{correct_flag_rate * 100:<18.2f}%"
    )

AnomalyFlow Score Threshold Analysis
Threshold   Flagged     Errors Caught   Error Recall   Correct Flagged   Correct Flag Rate 
----------------------------------------------------------------------------------------------------
0.10        150,377     34,717          97.06          %115,660           38.21             %
0.15        109,208     32,852          91.85          %76,356            25.23             %
0.20        82,162      30,086          84.12          %52,076            17.21             %
0.25        67,352      27,211          76.08          %40,141            13.26             %
0.30        54,656      23,696          66.25          %30,960            10.23             %
0.35        41,451      19,297          53.95          %22,154            7.32              %
0.40        28,720      14,307          40.00          %14,413            4.76              %
0.45        17,985      9,691           27.09          %8,294             2.74              %
0.50        9,163 

In [36]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 6: Prepare Validation DataLoader
# ==========================================================

X_val_scaled = scaler.transform(X_val)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
).unsqueeze(1)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

val_loader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False
)

print("Validation DataLoader created.")
print("Validation samples:", len(val_dataset))
print("Validation batches:", len(val_loader))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Validation DataLoader created.
Validation samples: 338444
Validation batches: 662


In [37]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 7: Generate Validation Predictions & Confidence
# ==========================================================

val_predictions = []
val_probabilities = []
val_true_labels = []

model.eval()

with torch.no_grad():

    for batch_X, batch_y in val_loader:

        batch_X = batch_X.to(device)

        logits = model(batch_X)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        confidence, predictions = torch.max(
            probabilities,
            dim=1
        )

        val_predictions.extend(
            predictions.cpu().numpy()
        )

        val_probabilities.extend(
            confidence.cpu().numpy()
        )

        val_true_labels.extend(
            batch_y.numpy()
        )

val_predictions = np.array(val_predictions)
val_probabilities = np.array(val_probabilities)
val_true_labels = np.array(val_true_labels)

print("Validation Predictions:", len(val_predictions))
print("Validation Confidence:", len(val_probabilities))
print("Validation True Labels:", len(val_true_labels))

val_accuracy = accuracy_score(
    val_true_labels,
    val_predictions
)

print(
    f"\nValidation Accuracy: "
    f"{val_accuracy * 100:.2f}%"
)

Validation Predictions: 338444
Validation Confidence: 338444
Validation True Labels: 338444

Validation Accuracy: 89.45%


In [38]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 8: Generate Validation Rényi Evidence
# ==========================================================

# Extract the original (unscaled) Rényi features
# from the validation split.

val_renyi_indices = [
    feature_columns.index(col)
    for col in renyi_columns
]

X_val_renyi = X_val[:, val_renyi_indices]

print("Validation Rényi features extracted.")
print("Shape:", X_val_renyi.shape)

Validation Rényi features extracted.
Shape: (338444, 8)


In [39]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 9: Calculate Validation Rényi Evidence
# ==========================================================

val_renyi_scaler = StandardScaler()

val_renyi_standardized = (
    val_renyi_scaler.fit_transform(
        X_val_renyi
    )
)

val_renyi_evidence = np.mean(
    np.abs(val_renyi_standardized),
    axis=1
)

print("Validation Rényi evidence generated.")
print("=" * 60)

print(
    f"Mean   : "
    f"{val_renyi_evidence.mean():.4f}"
)

print(
    f"Median : "
    f"{np.median(val_renyi_evidence):.4f}"
)

print(
    f"Minimum: "
    f"{val_renyi_evidence.min():.4f}"
)

print(
    f"Maximum: "
    f"{val_renyi_evidence.max():.4f}"
)

Validation Rényi evidence generated.
Mean   : 0.7663
Median : 0.6921
Minimum: 0.1905
Maximum: 4.2285


In [40]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 10: Prepare Validation Evidence Components
# ==========================================================

# Model uncertainty
val_confidence_uncertainty = (
    1.0 - val_probabilities
)

# Statistical evidence
val_statistical_evidence = (
    val_renyi_evidence
)

print("Validation evidence components prepared.")
print("=" * 70)

print(
    f"Confidence Uncertainty → "
    f"Mean: {val_confidence_uncertainty.mean():.4f}"
)

print(
    f"Rényi Evidence         → "
    f"Mean: {val_statistical_evidence.mean():.4f}"
)


Validation evidence components prepared.
Confidence Uncertainty → Mean: 0.1074
Rényi Evidence         → Mean: 0.7663


In [41]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 11: Normalize Validation Evidence
# ==========================================================

from sklearn.preprocessing import MinMaxScaler

# Normalize model uncertainty using validation data
val_confidence_scaler = MinMaxScaler()

val_confidence_normalized = (
    val_confidence_scaler
    .fit_transform(
        val_confidence_uncertainty.reshape(-1, 1)
    )
    .flatten()
)

# Normalize Rényi evidence using validation data
val_renyi_scaler_final = MinMaxScaler()

val_renyi_normalized = (
    val_renyi_scaler_final
    .fit_transform(
        val_statistical_evidence.reshape(-1, 1)
    )
    .flatten()
)

print("Validation evidence normalization completed.")
print("=" * 70)

print(
    f"Confidence Uncertainty → "
    f"Min: {val_confidence_normalized.min():.4f}, "
    f"Max: {val_confidence_normalized.max():.4f}"
)

print(
    f"Rényi Evidence         → "
    f"Min: {val_renyi_normalized.min():.4f}, "
    f"Max: {val_renyi_normalized.max():.4f}"
)

Validation evidence normalization completed.
Confidence Uncertainty → Min: 0.0000, Max: 1.0000
Rényi Evidence         → Min: 0.0000, Max: 1.0000


In [42]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 12: Generate Validation AnomalyFlow Score
# ==========================================================

# Initial equal weighting.
# We will evaluate this weighting before finalizing it.

CONFIDENCE_WEIGHT = 0.5
RENYI_WEIGHT = 0.5

val_anomalyflow_score = (
    CONFIDENCE_WEIGHT * val_confidence_normalized
    +
    RENYI_WEIGHT * val_renyi_normalized
)

print("Validation AnomalyFlow score generated.")
print("=" * 70)

print(
    f"Mean   : "
    f"{val_anomalyflow_score.mean():.4f}"
)

print(
    f"Median : "
    f"{np.median(val_anomalyflow_score):.4f}"
)

print(
    f"Minimum: "
    f"{val_anomalyflow_score.min():.4f}"
)

print(
    f"Maximum: "
    f"{val_anomalyflow_score.max():.4f}"
)

Validation AnomalyFlow score generated.
Mean   : 0.1469
Median : 0.0818
Minimum: 0.0066
Maximum: 0.6471


In [43]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 13: Evaluate Validation Score Separation
# ==========================================================

val_correct_mask = (
    val_predictions == val_true_labels
)

val_incorrect_mask = (
    val_predictions != val_true_labels
)

val_correct_scores = (
    val_anomalyflow_score[val_correct_mask]
)

val_incorrect_scores = (
    val_anomalyflow_score[val_incorrect_mask]
)

print("Validation AnomalyFlow Score by Prediction Correctness")
print("=" * 75)

print("\nCorrect Predictions")
print("-" * 75)

print(
    f"Samples         : "
    f"{len(val_correct_scores):,}"
)

print(
    f"Mean Score      : "
    f"{val_correct_scores.mean():.4f}"
)

print(
    f"Median Score    : "
    f"{np.median(val_correct_scores):.4f}"
)

print(
    f"75th Percentile : "
    f"{np.percentile(val_correct_scores, 75):.4f}"
)

print("\nIncorrect Predictions")
print("-" * 75)

print(
    f"Samples         : "
    f"{len(val_incorrect_scores):,}"
)

print(
    f"Mean Score      : "
    f"{val_incorrect_scores.mean():.4f}"
)

print(
    f"Median Score    : "
    f"{np.median(val_incorrect_scores):.4f}"
)

print(
    f"75th Percentile : "
    f"{np.percentile(val_incorrect_scores, 75):.4f}"
)

Validation AnomalyFlow Score by Prediction Correctness

Correct Predictions
---------------------------------------------------------------------------
Samples         : 302,752
Mean Score      : 0.1217
Median Score    : 0.0723
75th Percentile : 0.1439

Incorrect Predictions
---------------------------------------------------------------------------
Samples         : 35,692
Mean Score      : 0.3607
Median Score    : 0.3753
75th Percentile : 0.4679


In [44]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 14: Validation Threshold Analysis
# ==========================================================

validation_thresholds = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

total_val_incorrect = val_incorrect_mask.sum()
total_val_correct = val_correct_mask.sum()

print("Validation AnomalyFlow Threshold Analysis")
print("=" * 105)

print(
    f"{'Threshold':<12}"
    f"{'Flagged':<12}"
    f"{'Errors Caught':<16}"
    f"{'Error Recall':<16}"
    f"{'Correct Flagged':<18}"
    f"{'Correct Flag Rate':<18}"
)

print("-" * 105)

for threshold in validation_thresholds:

    flagged = (
        val_anomalyflow_score >= threshold
    )

    errors_caught = (
        flagged & val_incorrect_mask
    ).sum()

    correct_flagged = (
        flagged & val_correct_mask
    ).sum()

    error_recall = (
        errors_caught / total_val_incorrect
    )

    correct_flag_rate = (
        correct_flagged / total_val_correct
    )

    print(
        f"{threshold:<12.2f}"
        f"{flagged.sum():<12,}"
        f"{errors_caught:<16,}"
        f"{error_recall * 100:<16.2f}%"
        f"{correct_flagged:<18,}"
        f"{correct_flag_rate * 100:<18.2f}%"
    )

Validation AnomalyFlow Threshold Analysis
Threshold   Flagged     Errors Caught   Error Recall    Correct Flagged   Correct Flag Rate 
---------------------------------------------------------------------------------------------------------
0.10        144,375     34,619          96.99           %109,756           36.25             %
0.15        104,525     32,803          91.91           %71,722            23.69             %
0.20        81,725      30,196          84.60           %51,529            17.02             %
0.25        68,119      27,454          76.92           %40,665            13.43             %
0.30        55,969      24,225          67.87           %31,744            10.49             %
0.35        43,784      20,200          56.60           %23,584            7.79              %
0.40        30,929      15,290          42.84           %15,639            5.17              %
0.45        19,547      10,419          29.19           %9,128             3.02              %

In [45]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 15: Define Validation Decision Boundaries
# ==========================================================

# Decision boundaries derived from validation-set analysis.
#
# < 0.25  → Low anomaly evidence
# 0.25–<0.40 → Moderate anomaly evidence
# >= 0.40 → High anomaly evidence

LOW_THRESHOLD = 0.25
HIGH_THRESHOLD = 0.40

def classify_anomalyflow_score(score):

    if score < LOW_THRESHOLD:
        return "NORMAL"

    elif score < HIGH_THRESHOLD:
        return "SUSPICIOUS"

    else:
        return "HIGH_EVIDENCE"


val_decision_states = np.array([
    classify_anomalyflow_score(score)
    for score in val_anomalyflow_score
])

print("Validation Decision States")
print("=" * 60)

unique_states, state_counts = np.unique(
    val_decision_states,
    return_counts=True
)

for state, count in zip(
    unique_states,
    state_counts
):

    percentage = (
        count / len(val_decision_states)
    ) * 100

    print(
        f"{state:<15}: "
        f"{count:>8,} "
        f"({percentage:.2f}%)"
    )

Validation Decision States
HIGH_EVIDENCE  :   30,929 (9.14%)
NORMAL         :  270,325 (79.87%)
SUSPICIOUS     :   37,190 (10.99%)


In [46]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 16: Validate Decision States Against Baseline Errors
# ==========================================================

states = [
    "NORMAL",
    "SUSPICIOUS",
    "HIGH_EVIDENCE"
]

print("Decision State vs Baseline Prediction Correctness")
print("=" * 80)

for state in states:

    state_mask = (
        val_decision_states == state
    )

    state_total = state_mask.sum()

    state_errors = (
        state_mask & val_incorrect_mask
    ).sum()

    state_correct = (
        state_mask & val_correct_mask
    ).sum()

    error_rate = (
        state_errors / state_total * 100
        if state_total > 0
        else 0
    )

    print(f"\n{state}")
    print("-" * 80)

    print(
        f"Total samples : {state_total:,}"
    )

    print(
        f"Correct       : {state_correct:,}"
    )

    print(
        f"Baseline errors: {state_errors:,}"
    )

    print(
        f"Error rate    : {error_rate:.2f}%"
    )

Decision State vs Baseline Prediction Correctness

NORMAL
--------------------------------------------------------------------------------
Total samples : 270,325
Correct       : 262,087
Baseline errors: 8,238
Error rate    : 3.05%

SUSPICIOUS
--------------------------------------------------------------------------------
Total samples : 37,190
Correct       : 25,026
Baseline errors: 12,164
Error rate    : 32.71%

HIGH_EVIDENCE
--------------------------------------------------------------------------------
Total samples : 30,929
Correct       : 15,639
Baseline errors: 15,290
Error rate    : 49.44%


In [47]:
# ==========================================================
# Section 5: AnomalyFlow Decision Score
# Cell 17: Freeze Validation Configuration
# ==========================================================

anomalyflow_config = {
    "confidence_weight": CONFIDENCE_WEIGHT,
    "renyi_weight": RENYI_WEIGHT,
    "low_threshold": LOW_THRESHOLD,
    "high_threshold": HIGH_THRESHOLD,
    "states": {
        "low": "NORMAL",
        "medium": "SUSPICIOUS",
        "high": "HIGH_EVIDENCE"
    }
}

print("AnomalyFlow configuration frozen.")
print("=" * 60)

for key, value in anomalyflow_config.items():
    print(f"{key}: {value}")

AnomalyFlow configuration frozen.
confidence_weight: 0.5
renyi_weight: 0.5
low_threshold: 0.25
high_threshold: 0.4
states: {'low': 'NORMAL', 'medium': 'SUSPICIOUS', 'high': 'HIGH_EVIDENCE'}


# Section 6

In [48]:
# ==========================================================
# Section 6: Final AnomalyFlow Test Evaluation
# Cell 1: Apply Frozen Evidence Normalization to Test Set
# ==========================================================

# Apply the validation-fitted normalization parameters
# to the untouched test-set evidence.

test_confidence_normalized = (
    val_confidence_scaler
    .transform(
        confidence_uncertainty.reshape(-1, 1)
    )
    .flatten()
)

test_renyi_normalized = (
    val_renyi_scaler_final
    .transform(
        statistical_evidence.reshape(-1, 1)
    )
    .flatten()
)

print("Test evidence normalized using validation parameters.")
print("=" * 70)

print(
    f"Confidence Uncertainty → "
    f"Min: {test_confidence_normalized.min():.4f}, "
    f"Max: {test_confidence_normalized.max():.4f}"
)

print(
    f"Rényi Evidence         → "
    f"Min: {test_renyi_normalized.min():.4f}, "
    f"Max: {test_renyi_normalized.max():.4f}"
)

Test evidence normalized using validation parameters.
Confidence Uncertainty → Min: 0.0000, Max: 1.0734
Rényi Evidence         → Min: 0.0016, Max: 0.8996


In [49]:
# ==========================================================
# Section 6: Final AnomalyFlow Test Evaluation
# Cell 2: Generate Final AnomalyFlow Test Score
# ==========================================================

# Apply the frozen validation-derived weights.

test_anomalyflow_score = (
    CONFIDENCE_WEIGHT * test_confidence_normalized
    +
    RENYI_WEIGHT * test_renyi_normalized
)

print("Final test AnomalyFlow score generated.")
print("=" * 70)

print(
    f"Mean   : "
    f"{test_anomalyflow_score.mean():.4f}"
)

print(
    f"Median : "
    f"{np.median(test_anomalyflow_score):.4f}"
)

print(
    f"Minimum: "
    f"{test_anomalyflow_score.min():.4f}"
)

print(
    f"Maximum: "
    f"{test_anomalyflow_score.max():.4f}"
)

Final test AnomalyFlow score generated.
Mean   : 0.1469
Median : 0.0817
Minimum: 0.0069
Maximum: 0.6457


In [50]:
# ==========================================================
# Section 6: Final AnomalyFlow Test Evaluation
# Cell 3: Generate Final Test Decision States
# ==========================================================

test_decision_states = np.array([
    classify_anomalyflow_score(score)
    for score in test_anomalyflow_score
])

print("Final Test Decision States")
print("=" * 60)

unique_states, state_counts = np.unique(
    test_decision_states,
    return_counts=True
)

for state, count in zip(
    unique_states,
    state_counts
):

    percentage = (
        count / len(test_decision_states)
    ) * 100

    print(
        f"{state:<15}: "
        f"{count:>8,} "
        f"({percentage:.2f}%)"
    )

Final Test Decision States
HIGH_EVIDENCE  :   30,784 (9.10%)
NORMAL         :  270,361 (79.88%)
SUSPICIOUS     :   37,300 (11.02%)


In [51]:
# ==========================================================
# Section 6: Final AnomalyFlow Test Evaluation
# Cell 4: Evaluate Final Decision States
# ==========================================================

print("Final Test Decision State Evaluation")
print("=" * 80)

states = [
    "NORMAL",
    "SUSPICIOUS",
    "HIGH_EVIDENCE"
]

for state in states:

    state_mask = (
        test_decision_states == state
    )

    state_total = state_mask.sum()

    state_errors = (
        state_mask & incorrect_mask
    ).sum()

    state_correct = (
        state_mask & correct_mask
    ).sum()

    error_rate = (
        state_errors / state_total * 100
        if state_total > 0
        else 0
    )

    print(f"\n{state}")
    print("-" * 80)

    print(
        f"Total samples   : {state_total:,}"
    )

    print(
        f"Correct         : {state_correct:,}"
    )

    print(
        f"Baseline errors : {state_errors:,}"
    )

    print(
        f"Error rate      : {error_rate:.2f}%"
    )

Final Test Decision State Evaluation

NORMAL
--------------------------------------------------------------------------------
Total samples   : 270,361
Correct         : 262,113
Baseline errors : 8,248
Error rate      : 3.05%

SUSPICIOUS
--------------------------------------------------------------------------------
Total samples   : 37,300
Correct         : 25,038
Baseline errors : 12,262
Error rate      : 32.87%

HIGH_EVIDENCE
--------------------------------------------------------------------------------
Total samples   : 30,784
Correct         : 15,527
Baseline errors : 15,257
Error rate      : 49.56%


In [52]:
# ==========================================================
# Section 6: Final AnomalyFlow Test Evaluation
# Cell 5: Calculate Final Error Recall
# ==========================================================

# AnomalyFlow flags both SUSPICIOUS and HIGH_EVIDENCE
# samples as requiring additional attention.

flagged_mask = (
    (test_decision_states == "SUSPICIOUS")
    |
    (test_decision_states == "HIGH_EVIDENCE")
)

total_baseline_errors = incorrect_mask.sum()

errors_caught = (
    flagged_mask & incorrect_mask
).sum()

correctly_flagged = (
    flagged_mask & correct_mask
).sum()

error_recall = (
    errors_caught / total_baseline_errors
)

correct_flag_rate = (
    correctly_flagged / correct_mask.sum()
)

print("Final AnomalyFlow Error Detection")
print("=" * 70)

print(
    f"Total baseline errors : "
    f"{total_baseline_errors:,}"
)

print(
    f"Errors caught         : "
    f"{errors_caught:,}"
)

print(
    f"Error Recall          : "
    f"{error_recall * 100:.2f}%"
)

print(
    f"Correct predictions flagged : "
    f"{correctly_flagged:,}"
)

print(
    f"Correct Flag Rate     : "
    f"{correct_flag_rate * 100:.2f}%"
)

Final AnomalyFlow Error Detection
Total baseline errors : 35,767
Errors caught         : 27,519
Error Recall          : 76.94%
Correct predictions flagged : 40,565
Correct Flag Rate     : 13.40%


# Section 7

In [53]:
# ==========================================================
# Section 7: Security Severity & DDoS Interpretation
# Cell 1: Define DDoS and Non-DDoS Classes
# ==========================================================

# DDoS attack classes present in the dataset.
# These correspond to the distributed denial-of-service
# attack families represented in the training data.

ddos_classes = {
    "DrDoS_DNS",
    "DrDoS_LDAP",
    "DrDoS_MSSQL",
    "DrDoS_NTP",
    "DrDoS_NetBIOS",
    "DrDoS_SNMP",
    "DrDoS_SSDP",
    "DrDoS_UDP",
    "TFTP",
    "MSSQL",
    "NetBIOS",
    "Portmap",
    "Syn",
    "UDP",
    "UDP-lag"
}

# BENIGN and LDAP are treated as non-DDoS classes
# for the current security interpretation.
non_ddos_classes = {
    "BENIGN",
    "LDAP"
}

print("DDoS classes:", len(ddos_classes))
print("Non-DDoS classes:", len(non_ddos_classes))

print("\nDDoS Classes:")
for label in sorted(ddos_classes):
    print(" -", label)

print("\nNon-DDoS Classes:")
for label in sorted(non_ddos_classes):
    print(" -", label)

DDoS classes: 15
Non-DDoS classes: 2

DDoS Classes:
 - DrDoS_DNS
 - DrDoS_LDAP
 - DrDoS_MSSQL
 - DrDoS_NTP
 - DrDoS_NetBIOS
 - DrDoS_SNMP
 - DrDoS_SSDP
 - DrDoS_UDP
 - MSSQL
 - NetBIOS
 - Portmap
 - Syn
 - TFTP
 - UDP
 - UDP-lag

Non-DDoS Classes:
 - BENIGN
 - LDAP


In [54]:
# ==========================================================
# Section 7: Security Severity & DDoS Interpretation
# Cell 2: Map Predictions to DDoS Status
# ==========================================================

# Convert encoded predictions back to their original class names.

test_predicted_labels = label_encoder.inverse_transform(
    all_predictions
)

test_true_labels = label_encoder.inverse_transform(
    all_true_labels
)

# Determine whether each predicted class is DDoS-related.

predicted_ddos = np.array([
    label in ddos_classes
    for label in test_predicted_labels
])

actual_ddos = np.array([
    label in ddos_classes
    for label in test_true_labels
])

print("DDoS Interpretation Generated")
print("=" * 70)

print(
    f"Predicted DDoS samples     : "
    f"{predicted_ddos.sum():,}"
)

print(
    f"Predicted Non-DDoS samples : "
    f"{(~predicted_ddos).sum():,}"
)

print(
    f"Actual DDoS samples        : "
    f"{actual_ddos.sum():,}"
)

print(
    f"Actual Non-DDoS samples    : "
    f"{(~actual_ddos).sum():,}"
)

DDoS Interpretation Generated
Predicted DDoS samples     : 329,012
Predicted Non-DDoS samples : 9,433
Actual DDoS samples        : 328,632
Actual Non-DDoS samples    : 9,813


In [55]:
# ==========================================================
# Section 7: Security Severity & DDoS Interpretation
# Cell 3: Evaluate Binary DDoS Detection
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Binary DDoS evaluation
ddos_accuracy = accuracy_score(
    actual_ddos,
    predicted_ddos
)

ddos_precision = precision_score(
    actual_ddos,
    predicted_ddos,
    zero_division=0
)

ddos_recall = recall_score(
    actual_ddos,
    predicted_ddos,
    zero_division=0
)

ddos_f1 = f1_score(
    actual_ddos,
    predicted_ddos,
    zero_division=0
)

ddos_cm = confusion_matrix(
    actual_ddos,
    predicted_ddos
)

print("Binary DDoS Detection Performance")
print("=" * 70)

print(
    f"Accuracy  : {ddos_accuracy * 100:.2f}%"
)

print(
    f"Precision : {ddos_precision * 100:.2f}%"
)

print(
    f"Recall    : {ddos_recall * 100:.2f}%"
)

print(
    f"F1-Score  : {ddos_f1 * 100:.2f}%"
)

print("\nConfusion Matrix")
print("[[TN, FP],")
print(" [FN, TP]]")
print(ddos_cm)

Binary DDoS Detection Performance
Accuracy  : 99.58%
Precision : 99.73%
Recall    : 99.84%
F1-Score  : 99.78%

Confusion Matrix
[[TN, FP],
 [FN, TP]]
[[  8914    899]
 [   519 328113]]


In [56]:
# ==========================================================
# Section 7: Security Severity & DDoS Interpretation
# Cell 4: Analyze DDoS Distribution Across AnomalyFlow States
# ==========================================================

print("DDoS Distribution Across AnomalyFlow States")
print("=" * 85)

states = [
    "NORMAL",
    "SUSPICIOUS",
    "HIGH_EVIDENCE"
]

for state in states:

    state_mask = (
        test_decision_states == state
    )

    state_total = state_mask.sum()

    predicted_ddos_count = (
        state_mask & predicted_ddos
    ).sum()

    predicted_non_ddos_count = (
        state_mask & (~predicted_ddos)
    ).sum()

    ddos_percentage = (
        predicted_ddos_count / state_total * 100
        if state_total > 0
        else 0
    )

    print(f"\n{state}")
    print("-" * 85)

    print(
        f"Total samples       : "
        f"{state_total:,}"
    )

    print(
        f"Predicted DDoS      : "
        f"{predicted_ddos_count:,}"
    )

    print(
        f"Predicted Non-DDoS  : "
        f"{predicted_non_ddos_count:,}"
    )

    print(
        f"DDoS proportion     : "
        f"{ddos_percentage:.2f}%"
    )

DDoS Distribution Across AnomalyFlow States

NORMAL
-------------------------------------------------------------------------------------
Total samples       : 270,361
Predicted DDoS      : 261,669
Predicted Non-DDoS  : 8,692
DDoS proportion     : 96.79%

SUSPICIOUS
-------------------------------------------------------------------------------------
Total samples       : 37,300
Predicted DDoS      : 36,607
Predicted Non-DDoS  : 693
DDoS proportion     : 98.14%

HIGH_EVIDENCE
-------------------------------------------------------------------------------------
Total samples       : 30,784
Predicted DDoS      : 30,736
Predicted Non-DDoS  : 48
DDoS proportion     : 99.84%


In [57]:
# ==========================================================
# Section 7: Security Severity & DDoS Interpretation
# Cell 5: Actual DDoS Rate by AnomalyFlow State
# ==========================================================

print("Actual DDoS Rate Across AnomalyFlow States")
print("=" * 85)

for state in states:

    state_mask = (
        test_decision_states == state
    )

    state_total = state_mask.sum()

    actual_ddos_count = (
        state_mask & actual_ddos
    ).sum()

    actual_non_ddos_count = (
        state_mask & (~actual_ddos)
    ).sum()

    ddos_rate = (
        actual_ddos_count / state_total * 100
        if state_total > 0
        else 0
    )

    print(f"\n{state}")
    print("-" * 85)

    print(
        f"Total samples       : "
        f"{state_total:,}"
    )

    print(
        f"Actual DDoS         : "
        f"{actual_ddos_count:,}"
    )

    print(
        f"Actual Non-DDoS     : "
        f"{actual_non_ddos_count:,}"
    )

    print(
        f"Actual DDoS rate    : "
        f"{ddos_rate:.2f}%"
    )

Actual DDoS Rate Across AnomalyFlow States

NORMAL
-------------------------------------------------------------------------------------
Total samples       : 270,361
Actual DDoS         : 261,378
Actual Non-DDoS     : 8,983
Actual DDoS rate    : 96.68%

SUSPICIOUS
-------------------------------------------------------------------------------------
Total samples       : 37,300
Actual DDoS         : 36,512
Actual Non-DDoS     : 788
Actual DDoS rate    : 97.89%

HIGH_EVIDENCE
-------------------------------------------------------------------------------------
Total samples       : 30,784
Actual DDoS         : 30,742
Actual Non-DDoS     : 42
Actual DDoS rate    : 99.86%


In [58]:
# ==========================================================
# Section 7: Security Severity & DDoS Interpretation
# Cell 6: Create Security Severity Logic
# ==========================================================

def assign_security_severity(
    is_ddos,
    anomalyflow_state
):

    if not is_ddos:

        if anomalyflow_state == "NORMAL":
            return "NORMAL"

        elif anomalyflow_state == "SUSPICIOUS":
            return "MONITOR"

        else:
            return "INVESTIGATE"

    else:

        if anomalyflow_state == "NORMAL":
            return "LOW"

        elif anomalyflow_state == "SUSPICIOUS":
            return "MEDIUM"

        else:
            return "HIGH"


test_security_severity = np.array([
    assign_security_severity(
        predicted_ddos[i],
        test_decision_states[i]
    )
    for i in range(len(test_decision_states))
])

print("Security Severity Generated")
print("=" * 70)

unique_severities, severity_counts = np.unique(
    test_security_severity,
    return_counts=True
)

for severity, count in zip(
    unique_severities,
    severity_counts
):

    percentage = (
        count / len(test_security_severity)
    ) * 100

    print(
        f"{severity:<15}: "
        f"{count:>8,} "
        f"({percentage:.2f}%)"
    )

Security Severity Generated
HIGH           :   30,736 (9.08%)
INVESTIGATE    :       48 (0.01%)
LOW            :  261,669 (77.32%)
MEDIUM         :   36,607 (10.82%)
MONITOR        :      693 (0.20%)
NORMAL         :    8,692 (2.57%)


In [59]:
# ==========================================================
# Section 7: Security Severity & DDoS Interpretation
# Cell 7: Generate Final Security Decision Summary
# ==========================================================

print("=" * 80)
print("ANOMALYFLOW FINAL SECURITY DECISION SUMMARY")
print("=" * 80)

print("\n1. BASELINE MODEL")
print("-" * 80)

print(
    f"Test Accuracy        : "
    f"{baseline_accuracy * 100:.2f}%"
)

print(
    f"Test Samples         : "
    f"{len(all_true_labels):,}"
)

print(
    f"Correct Predictions  : "
    f"{correct_mask.sum():,}"
)

print(
    f"Baseline Errors      : "
    f"{incorrect_mask.sum():,}"
)

print("\n2. ANOMALYFLOW DECISION LAYER")
print("-" * 80)

print(
    f"Confidence Weight    : "
    f"{CONFIDENCE_WEIGHT}"
)

print(
    f"Rényi Weight         : "
    f"{RENYI_WEIGHT}"
)

print(
    f"Normal Threshold     : "
    f"< {LOW_THRESHOLD}"
)

print(
    f"High Threshold       : "
    f">= {HIGH_THRESHOLD}"
)

print(
    f"Error Recall         : "
    f"{error_recall * 100:.2f}%"
)

print(
    f"Correct Flag Rate    : "
    f"{correct_flag_rate * 100:.2f}%"
)

print("\n3. DECISION STATES")
print("-" * 80)

for state in states:

    count = (
        test_decision_states == state
    ).sum()

    percentage = (
        count / len(test_decision_states)
    ) * 100

    print(
        f"{state:<15}: "
        f"{count:>8,} "
        f"({percentage:.2f}%)"
    )

print("\n4. BINARY DDOS DETECTION")
print("-" * 80)

print(
    f"Accuracy           : "
    f"{ddos_accuracy * 100:.2f}%"
)

print(
    f"Precision          : "
    f"{ddos_precision * 100:.2f}%"
)

print(
    f"Recall             : "
    f"{ddos_recall * 100:.2f}%"
)

print(
    f"F1-Score           : "
    f"{ddos_f1 * 100:.2f}%"
)

print("\n5. SECURITY SEVERITY")
print("-" * 80)

for severity, count in zip(
    unique_severities,
    severity_counts
):

    percentage = (
        count / len(test_security_severity)
    ) * 100

    print(
        f"{severity:<15}: "
        f"{count:>8,} "
        f"({percentage:.2f}%)"
    )

print("\n" + "=" * 80)
print("FINAL SUMMARY GENERATED")
print("=" * 80)

ANOMALYFLOW FINAL SECURITY DECISION SUMMARY

1. BASELINE MODEL
--------------------------------------------------------------------------------
Test Accuracy        : 89.43%
Test Samples         : 338,445
Correct Predictions  : 302,678
Baseline Errors      : 35,767

2. ANOMALYFLOW DECISION LAYER
--------------------------------------------------------------------------------
Confidence Weight    : 0.5
Rényi Weight         : 0.5
Normal Threshold     : < 0.25
High Threshold       : >= 0.4
Error Recall         : 76.94%
Correct Flag Rate    : 13.40%

3. DECISION STATES
--------------------------------------------------------------------------------
NORMAL         :  270,361 (79.88%)
SUSPICIOUS     :   37,300 (11.02%)
HIGH_EVIDENCE  :   30,784 (9.10%)

4. BINARY DDOS DETECTION
--------------------------------------------------------------------------------
Accuracy           : 99.58%
Precision          : 99.73%
Recall             : 99.84%
F1-Score           : 99.78%

5. SECURITY SEVERITY
--

# Section 8


In [60]:
import json
import os

evaluation_dir = EVALUATION_DIR
os.makedirs(evaluation_dir, exist_ok=True)

anomalyflow_config_path = os.path.join(
    evaluation_dir,
    "anomalyflow_config.json"
)

with open(
    anomalyflow_config_path,
    "w"
) as f:

    json.dump(
        anomalyflow_config,
        f,
        indent=4
    )

print(
    "AnomalyFlow configuration saved to:"
)
print(anomalyflow_config_path)

AnomalyFlow configuration saved to:
/content/drive/MyDrive/Major_Project/models/evaluation/anomalyflow_config.json


In [61]:
# ==========================================================
# Section 8: Save AnomalyFlow Results
# Cell 2: Save Final Evaluation Summary
# ==========================================================

final_summary = {
    "baseline_model": {
        "test_accuracy": float(baseline_accuracy),
        "test_samples": int(len(all_true_labels)),
        "correct_predictions": int(correct_mask.sum()),
        "baseline_errors": int(incorrect_mask.sum())
    },

    "anomalyflow": {
        "confidence_weight": float(CONFIDENCE_WEIGHT),
        "renyi_weight": float(RENYI_WEIGHT),
        "low_threshold": float(LOW_THRESHOLD),
        "high_threshold": float(HIGH_THRESHOLD),
        "error_recall": float(error_recall),
        "correct_flag_rate": float(correct_flag_rate)
    },

    "decision_states": {
        state: {
            "count": int(
                (test_decision_states == state).sum()
            ),
            "percentage": float(
                (test_decision_states == state).mean() * 100
            )
        }
        for state in states
    },

    "binary_ddos_detection": {
        "accuracy": float(ddos_accuracy),
        "precision": float(ddos_precision),
        "recall": float(ddos_recall),
        "f1_score": float(ddos_f1),
        "confusion_matrix": ddos_cm.tolist()
    },

    "security_severity": {
        severity: {
            "count": int(count),
            "percentage": float(
                count / len(test_security_severity) * 100
            )
        }
        for severity, count in zip(
            unique_severities,
            severity_counts
        )
    }
}

summary_path = os.path.join(
    evaluation_dir,
    "anomalyflow_final_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        final_summary,
        f,
        indent=4
    )

print("Final AnomalyFlow evaluation summary saved to:")
print(summary_path)

Final AnomalyFlow evaluation summary saved to:
/content/drive/MyDrive/Major_Project/models/evaluation/anomalyflow_final_summary.json


In [62]:
# ==========================================================
# Section 8: Save AnomalyFlow Results
# Cell 3: Verify Saved Artifacts
# ==========================================================

print("Checking saved AnomalyFlow artifacts...")
print("=" * 60)

print(
    "Configuration exists:",
    os.path.exists(
        os.path.join(EVALUATION_DIR, "anomalyflow_config.json")
    )
)

print(
    "Final summary exists:",
    os.path.exists(
        os.path.join(EVALUATION_DIR, "anomalyflow_final_summary.json")
    )
)

print("\nSaved artifacts:")
print(
    " - " + os.path.join(EVALUATION_DIR, "anomalyflow_config.json")
)
print(
    " - " + os.path.join(EVALUATION_DIR, "anomalyflow_final_summary.json")
)

Checking saved AnomalyFlow artifacts...
Configuration exists: True
Final summary exists: True

Saved artifacts:
 - /content/drive/MyDrive/Major_Project/models/evaluation/anomalyflow_config.json
 - /content/drive/MyDrive/Major_Project/models/evaluation/anomalyflow_final_summary.json


In [63]:
import os

print("Current directory:", os.getcwd())

print("\nEvaluation directory:")
if os.path.exists(EVALUATION_DIR):
    print(os.listdir(EVALUATION_DIR))
else:
    print(f"Evaluation directory {EVALUATION_DIR} does not exist")

Current directory: /content

Evaluation directory:
['classification_report.json', 'confusion_matrix.npy', 'normalized_confusion_matrix.npy', 'evaluation_summary.json', 'anomalyflow_config.json', 'anomalyflow_final_summary.json']


# Section 9

In [64]:
# ==========================================================
# Section 9: Sample-Level Mitigation Export
# Cell 1: Inspect Existing Test-Sample Variables
# ==========================================================

# This cell ONLY inspects variables already present in the
# Notebook 06 runtime.
#
# It does NOT:
# - retrain the model
# - recompute predictions
# - reload the dataset
# - modify AnomalyFlow
# - modify thresholds or weights
# - modify existing evaluation results

EXPECTED_TEST_SAMPLES = 338_445

print("=" * 80)
print("INSPECTING EXISTING NOTEBOOK 06 TEST VARIABLES")
print("=" * 80)

# ----------------------------------------------------------
# Confirm the variables already established in Notebook 06
# ----------------------------------------------------------

confirmed_variables = {
    "Predicted labels": "all_predictions",
    "True labels": "all_true_labels",
    "Decision state": "test_decision_states",
    "Security severity": "test_security_severity"
}

print("\nConfirmed Notebook 06 variables:")
print("-" * 80)

for description, variable_name in confirmed_variables.items():

    exists = variable_name in globals()

    print(
        f"{description:<25} : "
        f"{variable_name:<25} "
        f"Exists = {exists}"
    )

    if exists:

        value = globals()[variable_name]

        try:
            print(
                f"    Type  : {type(value).__name__}"
            )
            print(
                f"    Length: {len(value):,}"
            )
            print(
                f"    Shape : "
                f"{getattr(value, 'shape', 'N/A')}"
            )

        except Exception:
            print("    Could not inspect shape/length.")


# ----------------------------------------------------------
# Inspect all existing variables that contain the expected
# test-sample length.
# ----------------------------------------------------------

print("\nVariables containing exactly 338,445 samples:")
print("-" * 80)

sample_length_variables = []

for variable_name, value in list(globals().items()):

    # Ignore internal/private variables
    if variable_name.startswith("_"):
        continue

    try:
        if len(value) == EXPECTED_TEST_SAMPLES:

            sample_length_variables.append(
                variable_name
            )

    except (TypeError, AttributeError):
        continue


for variable_name in sorted(
    sample_length_variables
):

    value = globals()[variable_name]

    print(
        f"{variable_name:<40} "
        f"type={type(value).__name__:<15} "
        f"shape={getattr(value, 'shape', 'N/A')}"
    )


print("\n" + "=" * 80)
print("INSPECTION COMPLETE")
print("=" * 80)

print(
    "\nDo not modify any variables based on this cell."
)

INSPECTING EXISTING NOTEBOOK 06 TEST VARIABLES

Confirmed Notebook 06 variables:
--------------------------------------------------------------------------------
Predicted labels          : all_predictions           Exists = True
    Type  : ndarray
    Length: 338,445
    Shape : (338445,)
True labels               : all_true_labels           Exists = True
    Type  : ndarray
    Length: 338,445
    Shape : (338445,)
Decision state            : test_decision_states      Exists = True
    Type  : ndarray
    Length: 338,445
    Shape : (338445,)
Security severity         : test_security_severity    Exists = True
    Type  : ndarray
    Length: 338,445
    Shape : (338445,)

Variables containing exactly 338,445 samples:
--------------------------------------------------------------------------------
X_test                                   type=ndarray         shape=(338445, 48)
X_test_renyi                             type=ndarray         shape=(338445, 8)
X_test_scaled                

In [65]:
# ==========================================================
# Section 9: Sample-Level Mitigation Export
# Cell 2: Prepare Sample-Level Results
# ==========================================================

import numpy as np
import pandas as pd
import os
import json

EXPECTED_TEST_SAMPLES = 338_445

# ----------------------------------------------------------
# Use the FINAL test-set variables already generated by
# Notebook 06.
# ----------------------------------------------------------

sample_results = pd.DataFrame({
    "Test_Index": np.arange(EXPECTED_TEST_SAMPLES),

    "True_Label": np.asarray(
        test_true_labels
    ),

    "Predicted_Label": np.asarray(
        test_predicted_labels
    ),

    "Confidence": np.asarray(
        confidence_values
    ).reshape(-1),

    "Confidence_Uncertainty": np.asarray(
        confidence_uncertainty
    ).reshape(-1),

    "Renyi_Evidence": np.asarray(
        renyi_evidence_score
    ).reshape(-1),

    "Confidence_Normalized": np.asarray(
        test_confidence_normalized
    ).reshape(-1),

    "Renyi_Normalized": np.asarray(
        test_renyi_normalized
    ).reshape(-1),

    "AnomalyFlow_Score": np.asarray(
        test_anomalyflow_score
    ).reshape(-1),

    "Decision_State": np.asarray(
        test_decision_states
    ),

    "Security_Severity": np.asarray(
        test_security_severity
    )
})

print("Sample-level results prepared.")
print("=" * 80)

print(
    f"Rows    : {len(sample_results):,}"
)

print(
    f"Columns : {len(sample_results.columns)}"
)

print("\nColumns:")
for column in sample_results.columns:
    print(f" - {column}")

print("\nFirst 5 rows:")
display(sample_results.head())

Sample-level results prepared.
Rows    : 338,445
Columns : 11

Columns:
 - Test_Index
 - True_Label
 - Predicted_Label
 - Confidence
 - Confidence_Uncertainty
 - Renyi_Evidence
 - Confidence_Normalized
 - Renyi_Normalized
 - AnomalyFlow_Score
 - Decision_State
 - Security_Severity

First 5 rows:


,Test_Index,True_Label,Predicted_Label,Confidence,Confidence_Uncertainty,Renyi_Evidence,Confidence_Normalized,Renyi_Normalized,AnomalyFlow_Score,Decision_State,Security_Severity
0,0,NetBIOS,NetBIOS,0.274310,0.274310,0.417463,0.386017,0.056216,0.221116,NORMAL,LOW
1,1,TFTP,TFTP,0.000000,0.000000,0.714693,0.000000,0.129823,0.064911,NORMAL,LOW
2,2,Syn,Syn,0.000000,0.000000,1.309719,0.000000,0.277177,0.138588,NORMAL,LOW
3,3,TFTP,TFTP,0.000000,0.000000,0.627796,0.000000,0.108303,0.054152,NORMAL,LOW
4,4,DrDoS_SSDP,DrDoS_UDP,0.628116,0.628116,1.375605,0.883902,0.293493,0.588697,HIGH_EVIDENCE,HIGH


In [66]:
# ==========================================================
# Section 9: Sample-Level Mitigation Export
# Cell 3: Save Sample-Level CSV
# ==========================================================

mitigation_dir = (
    "/content/drive/MyDrive/Major_Project/mitigation"
)

os.makedirs(
    mitigation_dir,
    exist_ok=True
)

sample_output_path = os.path.join(
    mitigation_dir,
    "anomalyflow_test_results.csv"
)

sample_results.to_csv(
    sample_output_path,
    index=False
)

print("Sample-level AnomalyFlow results saved.")
print("=" * 80)

print(
    f"Output path : {sample_output_path}"
)

print(
    f"Rows        : {len(sample_results):,}"
)

print(
    f"Columns     : {len(sample_results.columns)}"
)

print(
    f"File size   : "
    f"{os.path.getsize(sample_output_path) / (1024 ** 2):.2f} MB"
)

Sample-level AnomalyFlow results saved.
Output path : /content/drive/MyDrive/Major_Project/mitigation/anomalyflow_test_results.csv
Rows        : 338,445
Columns     : 11
File size   : 38.15 MB


In [67]:
# ==========================================================
# Section 9: Sample-Level Mitigation Export
# Cell 4: Save Mitigation Input Summary
# ==========================================================

# ----------------------------------------------------------
# Build compact summary from the FINAL test-level results.
# No predictions or AnomalyFlow values are recalculated.
# ----------------------------------------------------------

decision_state_counts = (
    sample_results["Decision_State"]
    .value_counts()
    .to_dict()
)

security_severity_counts = (
    sample_results["Security_Severity"]
    .value_counts()
    .to_dict()
)

anomalyflow_score_statistics = {
    "mean": float(
        sample_results["AnomalyFlow_Score"].mean()
    ),
    "median": float(
        sample_results["AnomalyFlow_Score"].median()
    ),
    "minimum": float(
        sample_results["AnomalyFlow_Score"].min()
    ),
    "maximum": float(
        sample_results["AnomalyFlow_Score"].max()
    )
}

mitigation_summary = {
    "total_test_samples": int(
        len(sample_results)
    ),

    "decision_state_counts": {
        str(key): int(value)
        for key, value in decision_state_counts.items()
    },

    "security_severity_counts": {
        str(key): int(value)
        for key, value in security_severity_counts.items()
    },

    "baseline_accuracy": float(
        baseline_accuracy
    ),

    "anomalyflow_score_statistics":
        anomalyflow_score_statistics,

    "sample_level_output_file":
        sample_output_path
}

summary_output_path = os.path.join(
    mitigation_dir,
    "mitigation_input_summary.json"
)

with open(
    summary_output_path,
    "w"
) as f:

    json.dump(
        mitigation_summary,
        f,
        indent=4
    )

print("Mitigation input summary saved.")
print("=" * 80)

print(
    f"Output path : {summary_output_path}"
)

print("\nDecision-state counts:")
for state, count in decision_state_counts.items():
    print(
        f" - {state}: {count:,}"
    )

print("\nSecurity-severity counts:")
for severity, count in security_severity_counts.items():
    print(
        f" - {severity}: {count:,}"
    )

print("\nAnomalyFlow score statistics:")
for key, value in anomalyflow_score_statistics.items():
    print(
        f" - {key.capitalize():<8}: {value:.4f}"
    )

Mitigation input summary saved.
Output path : /content/drive/MyDrive/Major_Project/mitigation/mitigation_input_summary.json

Decision-state counts:
 - NORMAL: 270,361
 - SUSPICIOUS: 37,300
 - HIGH_EVIDENCE: 30,784

Security-severity counts:
 - LOW: 261,669
 - MEDIUM: 36,607
 - HIGH: 30,736
 - NORMAL: 8,692
 - MONITOR: 693
 - INVESTIGATE: 48

AnomalyFlow score statistics:
 - Mean    : 0.1469
 - Median  : 0.0817
 - Minimum : 0.0069
 - Maximum : 0.6457


In [68]:
# ==========================================================
# Section 9: Sample-Level Mitigation Export
# Cell 5: Final Artifact Verification
# ==========================================================

print("=" * 80)
print("FINAL SAMPLE-LEVEL ARTIFACT VERIFICATION")
print("=" * 80)

# ----------------------------------------------------------
# Expected values from the FINAL Notebook 06 results
# ----------------------------------------------------------

EXPECTED_ROWS = 338_445

EXPECTED_DECISION_STATES = {
    "NORMAL": 270_361,
    "SUSPICIOUS": 37_300,
    "HIGH_EVIDENCE": 30_784
}

EXPECTED_SECURITY_SEVERITIES = {
    "LOW": 261_669,
    "MEDIUM": 36_607,
    "HIGH": 30_736,
    "NORMAL": 8_692,
    "MONITOR": 693,
    "INVESTIGATE": 48
}

REQUIRED_COLUMNS = [
    "Test_Index",
    "True_Label",
    "Predicted_Label",
    "Confidence",
    "Confidence_Uncertainty",
    "Renyi_Evidence",
    "Confidence_Normalized",
    "Renyi_Normalized",
    "AnomalyFlow_Score",
    "Decision_State",
    "Security_Severity"
]

EXPECTED_DECISION_VALUES = {
    "NORMAL",
    "SUSPICIOUS",
    "HIGH_EVIDENCE"
}

EXPECTED_SEVERITY_VALUES = {
    "NORMAL",
    "MONITOR",
    "INVESTIGATE",
    "LOW",
    "MEDIUM",
    "HIGH"
}


# ----------------------------------------------------------
# 1. Verify CSV exists and load it
# ----------------------------------------------------------

assert os.path.exists(
    sample_output_path
), "Sample-level CSV was not found."

saved_results = pd.read_csv(
    sample_output_path
)

print("\n1. CSV File")
print("-" * 80)

print(
    f"File exists : True"
)

print(
    f"Rows        : {len(saved_results):,}"
)


# ----------------------------------------------------------
# 2. Verify exact row count
# ----------------------------------------------------------

rows_correct = (
    len(saved_results) == EXPECTED_ROWS
)

print("\n2. Row Count")
print("-" * 80)

print(
    f"Expected : {EXPECTED_ROWS:,}"
)

print(
    f"Actual   : {len(saved_results):,}"
)

print(
    f"PASS     : {rows_correct}"
)


# ----------------------------------------------------------
# 3. Verify required columns
# ----------------------------------------------------------

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in saved_results.columns
]

columns_correct = (
    len(missing_columns) == 0
)

print("\n3. Required Columns")
print("-" * 80)

print(
    f"Required columns : {len(REQUIRED_COLUMNS)}"
)

print(
    f"Missing columns  : {missing_columns}"
)

print(
    f"PASS             : {columns_correct}"
)


# ----------------------------------------------------------
# 4. Verify no missing values in required fields
# ----------------------------------------------------------

required_missing_values = (
    saved_results[
        REQUIRED_COLUMNS
    ]
    .isna()
    .sum()
)

total_required_missing = int(
    required_missing_values.sum()
)

missing_values_correct = (
    total_required_missing == 0
)

print("\n4. Missing Values")
print("-" * 80)

print(
    f"Missing required values : "
    f"{total_required_missing:,}"
)

print(
    f"PASS                    : "
    f"{missing_values_correct}"
)


# ----------------------------------------------------------
# 5. Verify Decision_State values
# ----------------------------------------------------------

actual_decision_values = set(
    saved_results[
        "Decision_State"
    ].unique()
)

decision_values_correct = (
    actual_decision_values
    .issubset(EXPECTED_DECISION_VALUES)
)

print("\n5. Decision State Values")
print("-" * 80)

print(
    "Allowed:"
)

for value in sorted(
    EXPECTED_DECISION_VALUES
):
    print(f" - {value}")

print(
    "\nActual:"
)

for value in sorted(
    actual_decision_values
):
    print(f" - {value}")

print(
    f"\nPASS : {decision_values_correct}"
)


# ----------------------------------------------------------
# 6. Verify Security_Severity values
# ----------------------------------------------------------

actual_severity_values = set(
    saved_results[
        "Security_Severity"
    ].unique()
)

severity_values_correct = (
    actual_severity_values
    .issubset(EXPECTED_SEVERITY_VALUES)
)

print("\n6. Security Severity Values")
print("-" * 80)

print(
    "Allowed:"
)

for value in sorted(
    EXPECTED_SEVERITY_VALUES
):
    print(f" - {value}")

print(
    "\nActual:"
)

for value in sorted(
    actual_severity_values
):
    print(f" - {value}")

print(
    f"\nPASS : {severity_values_correct}"
)


# ----------------------------------------------------------
# 7. Compare Decision_State counts with Notebook 06
# ----------------------------------------------------------

actual_decision_counts = (
    saved_results[
        "Decision_State"
    ]
    .value_counts()
    .to_dict()
)

decision_counts_correct = (
    actual_decision_counts
    == EXPECTED_DECISION_STATES
)

print("\n7. Decision-State Count Verification")
print("-" * 80)

print("Expected:")
for state, count in EXPECTED_DECISION_STATES.items():
    print(
        f" - {state:<15}: {count:,}"
    )

print("\nSaved CSV:")
for state, count in actual_decision_counts.items():
    print(
        f" - {state:<15}: {count:,}"
    )

print(
    f"\nPASS : {decision_counts_correct}"
)


# ----------------------------------------------------------
# 8. Compare Security_Severity counts with Notebook 06
# ----------------------------------------------------------

actual_severity_counts = (
    saved_results[
        "Security_Severity"
    ]
    .value_counts()
    .to_dict()
)

severity_counts_correct = (
    actual_severity_counts
    == EXPECTED_SECURITY_SEVERITIES
)

print("\n8. Security-Severity Count Verification")
print("-" * 80)

print("Expected:")
for severity, count in EXPECTED_SECURITY_SEVERITIES.items():
    print(
        f" - {severity:<15}: {count:,}"
    )

print("\nSaved CSV:")
for severity, count in actual_severity_counts.items():
    print(
        f" - {severity:<15}: {count:,}"
    )

print(
    f"\nPASS : {severity_counts_correct}"
)


# ----------------------------------------------------------
# 9. Verify Test_Index uniqueness
# ----------------------------------------------------------

test_index_unique = (
    saved_results[
        "Test_Index"
    ].nunique()
    == EXPECTED_ROWS
)

test_index_range_correct = (
    saved_results[
        "Test_Index"
    ].min() == 0
    and
    saved_results[
        "Test_Index"
    ].max() == EXPECTED_ROWS - 1
)

print("\n9. Test Index Verification")
print("-" * 80)

print(
    f"Unique indices : "
    f"{saved_results['Test_Index'].nunique():,}"
)

print(
    f"Minimum index  : "
    f"{saved_results['Test_Index'].min()}"
)

print(
    f"Maximum index  : "
    f"{saved_results['Test_Index'].max()}"
)

print(
    f"Unique PASS    : {test_index_unique}"
)

print(
    f"Range PASS     : {test_index_range_correct}"
)


# ----------------------------------------------------------
# 10. Final verification
# ----------------------------------------------------------

all_checks_passed = all([
    rows_correct,
    columns_correct,
    missing_values_correct,
    decision_values_correct,
    severity_values_correct,
    decision_counts_correct,
    severity_counts_correct,
    test_index_unique,
    test_index_range_correct
])

print("\n" + "=" * 80)

if all_checks_passed:

    print("ALL SAMPLE-LEVEL ARTIFACT CHECKS PASSED")
    print("=" * 80)

    print(
        "\nNotebook 07 input is ready."
    )

    print(
        f"\nCSV:"
    )

    print(
        sample_output_path
    )

    print(
        "\nJSON:"
    )

    print(
        summary_output_path
    )

else:

    print("SOME ARTIFACT CHECKS FAILED")
    print("=" * 80)

    raise RuntimeError(
        "Sample-level mitigation artifact verification failed."
    )

FINAL SAMPLE-LEVEL ARTIFACT VERIFICATION

1. CSV File
--------------------------------------------------------------------------------
File exists : True
Rows        : 338,445

2. Row Count
--------------------------------------------------------------------------------
Expected : 338,445
Actual   : 338,445
PASS     : True

3. Required Columns
--------------------------------------------------------------------------------
Required columns : 11
Missing columns  : []
PASS             : True

4. Missing Values
--------------------------------------------------------------------------------
Missing required values : 0
PASS                    : True

5. Decision State Values
--------------------------------------------------------------------------------
Allowed:
 - HIGH_EVIDENCE
 - NORMAL
 - SUSPICIOUS

Actual:
 - HIGH_EVIDENCE
 - NORMAL
 - SUSPICIOUS

PASS : True

6. Security Severity Values
--------------------------------------------------------------------------------
Allowed:
 - HIGH